# Next-Concept Modeling for mazes

Train one NCM tokenizer/generator pair, then evaluate **NCM alone** on 32×32 and 64×64 grids. Measure solvability, shortest solution length, A* expansions, reachable passage space, and isolated components.

Based on [Next-Concept Modeling](https://arxiv.org/abs/2610.14162) and the [authors' code](https://github.com/HazyResearch/ncm/tree/c70274273f489abc73832b2b98584a8483b6b203). This is a 2D adaptation with spatial scales and coordinates. Total capacity remains about 4.31M parameters, but the training budget is expanded to allow convergence.

Run setup, data, initialization, checkpoint loading, and training-target configuration. Both tokenizer and generator train from scratch for their complete update targets; resuming a new run is an explicit option. No CFM checkpoints are loaded or evaluated.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import hashlib
import html
import json
import math
import numpy as np
import torch
import plotly.graph_objects as go
from IPython.display import HTML, display
from torch.nn import functional as F
from torch.utils.data import Dataset, TensorDataset
from _shared.model import Denoiser

import importlib
import _shared.ncm as ncm_module
import _shared.ncm_training as ncm_training_module
import _shared.ncm_benchmarks as ncm_benchmarks_module

importlib.reload(ncm_module)
importlib.reload(ncm_training_module)
importlib.reload(ncm_benchmarks_module)

from _shared.ncm import NCMConfig, GridNCM, matched_config, PAPER, REFERENCE
from _shared.ncm_training import NCMRun, train_online, file_sha256
from _shared.ncm_benchmarks import board_panels, PLOT_CONFIG, progress_bar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
ARTIFACTS = PROJECT_ROOT / "artifacts" / "ncm_mazes"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "maze_ncm.pt"
from _shared.maze_data import generate_dataset, solvability_rate
from _shared.ncm_benchmarks import evaluate_mazes, plot_maze_metrics

## 1. The same training mazes

Generate the same 12,000 DFS mazes deterministically with seed 7: the first 10,800 are used for training and the last 1,200 for validation. `0` is a wall; `1` is a passage. The model is **not fine-tuned** at 64×64.


In [2]:
MAZE_SIZE = 32
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED, size=MAZE_SIZE)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
train_dataset = TensorDataset(train_mazes)
validation_dataset = TensorDataset(val_mazes)
data_signature = dict(
    task="mazes",
    size=MAZE_SIZE,
    seed=SEED,
    dataset_sha256=hashlib.sha256(mazes.tobytes()).hexdigest(),
    train_count=len(train_mazes),
    validation_count=len(val_mazes),
)
board_panels(
    mazes[:4], [f"Train maze {i+1}" for i in range(4)], "Training mazes", columns=4
).show(config=PLOT_CONFIG)

## 2. Architecture and parameter count

Scales: **1×1 → 2×2 → 4×4 → 8×8 → 16×16 → 32×32**. Each level has its own codebook of 64 vectors with dimension 64. The encoder combines class embeddings with 2D positions; the decoder is one bidirectional transformer block. The generator is a small shared transformer with a separate output head for each codebook.

Generator width and depth are selected automatically **by parameter count alone**, without evaluating test quality. The total includes the tokenizer, decoder, EMA codebooks, and generator; the target is the original two-channel U-Net's 4,315,458 parameters.

At 64×64, the same codebooks and heads operate at scales 1, 4, 8, 16, 32, and 64. The first-level global operator resizes to the training resolution and back. This evaluates resolution transfer without training at 64×64.


In [3]:
BATCH_SIZE = 128
MICRO_BATCH_SIZE = 32  # Reduce to 16/8 if GPU memory is limited.
with torch.device("meta"):
    baseline_parameters = sum(p.numel() for p in Denoiser(embedding_dim=2).parameters())
proposal = NCMConfig(
    size=MAZE_SIZE, classes=2, scales=(1, 2, 4, 8, 16, 32), codebook_size=64
)
config, ncm_parameters = matched_config(proposal, baseline_parameters)
run = NCMRun(
    config,
    CHECKPOINT_PATH,
    data_signature=data_signature,
    seed=SEED,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    micro_batch_size=MICRO_BATCH_SIZE,
)
counts = run.model.parameter_counts()
display(
    HTML(
        f"<b>NCM:</b> {counts['total']:,} = tokenizer {counts['tokenizer']:,} + generator {counts['generator']:,}<br>"
        f"<b>U-Net:</b> {baseline_parameters:,} · difference {100*(counts['total']/baseline_parameters-1):+.2f}%<br>"
        f"Generator: width {config.width}, {config.layers} blocks · device {DEVICE}"
    )
)

### Fresh run or explicit resume

By default, both the tokenizer and generator start from random initialization.
Set `RESUME_CHECKPOINT = True` only to resume a checkpoint from this new run after interruption.
Run setup and model initialization before this cell. Loading restores both stages,
optimizer states, RNG state, and history; it does not shorten the tokenizer's target.


In [4]:
RESUME_CHECKPOINT = False
if RESUME_CHECKPOINT:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            "No NCM checkpoint to resume. Use RESUME_CHECKPOINT = False for a fresh run."
        )
    run.load()
    display(
        HTML(
            "<b>Resuming NCM:</b> "
            + " · ".join(
                f"{stage}: {state['updates']:,} updates"
                for stage, state in run.state.items()
            )
        )
    )
else:
    if any(state["updates"] for state in run.state.values()):
        raise RuntimeError(
            "Re-run the model initialization cell to start both stages from scratch."
        )
    display(
        HTML("<b>Fresh NCM run:</b> tokenizer and generator initialized from scratch.")
    )

## 3. Full training budgets for both stages

Train **both stages from scratch**. Stage I always runs its full update target,
even if validation pixel or whole-grid accuracy reaches 100% early. Reconstruction
accuracy alone is not used to skip training or freeze an unfinished tokenizer.

Defaults: **mazes — 1,000 tokenizer + 2,000 generator updates**;
**Queens — 2,000 tokenizer + 20,000 generator updates**. Targets are total optimizer
updates for this run. These expanded budgets do not claim equal training time with
earlier baselines. Actual elapsed time and data exposure are recorded.

Stage II is allowed only after the complete Stage I budget has been trained.
Then check exact whole-grid reconstruction on 128 validation examples and freeze
the tokenizer. If it fails the readiness check, extend Stage I before starting
the generator. The hierarchy cannot be changed after generator training begins.

The generator uses BF16 on supported CUDA devices, otherwise FP32. Tokenizer,
code assignment, and decoder remain FP32. Warmup precedes cosine decay over the
configured update count. Explicit resume continues these same full targets.


In [5]:
TOKENIZER_TARGET_UPDATES = 1000
GENERATOR_TARGET_UPDATES = 2000
TOKENIZER_MIN_EXACT = 0.99
GENERATOR_PRECISION = "auto"
PACK_LEVELS = False
LEARNING_RATE = 2e-4

run.micro_batch_size = MICRO_BATCH_SIZE
run.learning_rate = LEARNING_RATE
run.set_training_targets(
    tokenizer_updates=TOKENIZER_TARGET_UPDATES,
    generator_updates=GENERATOR_TARGET_UPDATES,
    precision=GENERATOR_PRECISION,
    pack_levels=PACK_LEVELS,
    warmup_updates=100,
)
display(
    HTML(
        f"<b>Full update targets:</b> tokenizer {TOKENIZER_TARGET_UPDATES:,}; generator {GENERATOR_TARGET_UPDATES:,}<br>"
        f"Completed: tokenizer {run.state['tokenizer']['updates']:,}; generator {run.state['generator']['updates']:,}<br>"
        f"Generator precision: {run.model.generator_precision} · effective batch {run.batch_size} · microbatch {run.micro_batch_size}"
    )
)

## 4. Stage I — learn the code hierarchy

For a target grid $x$, the encoder produces $z=E_\phi(x)$. At each scale, quantize the residual left by the preceding levels:

$$
r_k=\operatorname{VQ}_k\!\left(\operatorname{Down}_k(z-\widehat z^{(k-1)})\right),
\qquad
\widehat z^{(k)}=\widehat z^{(k-1)}+R_k(e_k(r_k)).
$$

With probability $0.5$, retain a random incomplete prefix of levels; otherwise, retain all levels. The loss at the selected cutoff $k$ follows Equation (9) of the paper:

$$
\mathcal L_{\mathrm{tok}}=
\operatorname{CE}(x,D_\psi(\widehat z^{(k)}))
+0.25\,\operatorname{MSE}(z,\operatorname{sg}[\widehat z^{(k)}])
+\operatorname{MSE}(\operatorname{sg}[z],\widehat z^{(k)}).
$$

A straight-through estimator handles discrete selection; codebooks use EMA updates, following the authors' implementation. The learned first-level operator preserves global information, while residual convolutions after upsampling add detail. Validation always uses **all** levels, so its objective can differ from training with nested dropout.


In [6]:
import importlib
import _shared.ncm_training as ncm_training

train_online = importlib.reload(ncm_training).train_online

if run.state["tokenizer"]["updates"] < run.plan["tokenizer_updates"]:
    train_online(run, "tokenizer", train_dataset, validation_dataset, report_every=50)
else:
    display(HTML("The full tokenizer update target has been completed."))

### 4.1 What does quantization lose?

Show reconstructions of **one validation maze** after each level. This encodes a known target. Also measure pixel accuracy, the fraction of exactly reconstructed validation grids, and their solvability. Tokenizer errors alone can break connectivity; inspect this reconstruction ceiling separately from generator errors.


In [7]:
if run.state["tokenizer"]["updates"] == 0:
    raise RuntimeError("Train or load the tokenizer first.")
run.model.eval()
with torch.no_grad():
    example = val_mazes[:1].to(DEVICE)
    _, partial = run.model.tokenizer.quantize(run.model.tokenizer.latent(example))
    reconstructions = [
        run.model.tokenizer.decode(z).argmax(1)[0].cpu().numpy() for z in partial
    ]
    board_panels(
        [example[0].cpu().numpy(), *reconstructions],
        ["Target", *[f"Level {i+1}: {s}×{s}" for i, s in enumerate(config.scales)]],
        "Tokenizer · target reconstruction",
        columns=4,
    ).show(config=PLOT_CONFIG)
    validation_target = val_mazes[:128]
    restored = []
    for start in range(0, len(validation_target), MICRO_BATCH_SIZE):
        target = validation_target[start : start + MICRO_BATCH_SIZE].to(DEVICE)
        _, reconstruction = run.model.tokenizer.quantize(
            run.model.tokenizer.latent(target)
        )
        restored.append(run.model.tokenizer.decode(reconstruction[-1]).argmax(1).cpu())
    restored = torch.cat(restored)
display(
    HTML(
        f"Validation pixel accuracy: {100*(restored == validation_target).float().mean():.2f}%<br>"
        f"Exact grids: {100*(restored == validation_target).flatten(1).all(1).float().mean():.2f}%<br>"
        f"Solvable reconstructions: {100*solvability_rate(restored.numpy()):.2f}%"
    )
)

## 5. Stage II — predict the next level

Freeze the entire tokenizer. The generator receives only the accumulated reconstruction of **preceding** levels, resized to the next level, along with positions and the level index. Ground-truth codes for the current level are never provided as input.

$$
p_\theta(r_{1:K})=\prod_{k=1}^{K}\prod_i
p_\theta(r_k^i\mid\widehat z^{(k-1)}),
\qquad
\mathcal L_{\mathrm{NCM}}=\frac1K\sum_{k=1}^{K}
\operatorname{CE}_{\mathrm{mean}}(r_k,\mathrm{logits}_k).
$$

This is Equation (10) with $\alpha=1$: each scale receives equal weight. During training, some **conditioning codes** are replaced by random codes from the same codebook; the replacement probability increases linearly from 0 to 0.1 across levels. Labels remain clean. This is a local hyperparameter choice, not the paper's Sudoku setting.

For these large grids, separate level blocks avoid computing masked cross-level attention. This is equivalent to the packed block-diagonal objective and was faster on the measured GPU. Gradients from all levels and microbatches accumulate before **one** AdamW update. During generation, levels are sampled sequentially from the model's own predictions.


In [8]:
import importlib
import _shared.ncm_training as ncm_training

train_online = importlib.reload(ncm_training).train_online

if run.state["tokenizer"]["updates"] < run.plan["tokenizer_updates"]:
    raise RuntimeError(
        "Complete the full tokenizer training cell before starting the generator."
    )

tokenizer_quality = run.tokenizer_quality(validation_dataset, examples=128)
if tokenizer_quality["exact_accuracy"] < TOKENIZER_MIN_EXACT:
    raise RuntimeError(
        "Tokenizer reconstruction is not ready for generation. Increase TOKENIZER_TARGET_UPDATES, "
        "rerun the targets cell, then continue stage I before training the generator."
    )
train_online(run, "generator", train_dataset, validation_dataset, report_every=100)

### 5.1 Saving and actual training cost

Both stages are saved automatically when the plot updates and when training is interrupted. This cell also writes an explicit checkpoint and a JSON budget audit. Rerunning training continues the configured total update target. Per-level validation CE reveals whether fine scales have learned.


In [9]:
run.save()
audit = run.report()
actual_seconds = sum(s["seconds"] for s in run.state.values())
display(
    HTML(
        f"<b>One NCM run</b> · tokenizer + generator: {audit['parameters']['total']:,} parameters<br>"
        f"Actual total training: {actual_seconds/60:.2f} min · expanded update budget<br>"
        + "<br>".join(
            f"{name}: {s['updates']:,} / {run.plan[name+'_updates']:,} updates, "
            f"{s['examples']:,} examples, {s['seconds']/60:.2f} min"
            for name, s in run.state.items()
        )
        + f"<br>Checkpoint: <code>{html.escape(str(run.path.relative_to(PROJECT_ROOT)))}</code>"
    )
)

# A scalar mean can hide unlearned fine levels. Inspect validation CE by scale.
level_history = [
    h for h in run.history["generator"] if h.get("validation_level_ce") is not None
]
if level_history:
    level_fig = go.Figure()
    for k, size in enumerate(config.scales):
        level_fig.add_scatter(
            x=[h["update"] for h in level_history],
            y=[h["validation_level_ce"][k] for h in level_history],
            name=f"Level {k+1} · {size}×{size}",
            mode="lines+markers",
        )
    level_fig.update_layout(
        template="plotly_white",
        title="NCM · validation CE by level",
        xaxis_title="Generator updates",
        yaxis_title="Code cross-entropy (nats)",
        height=430,
        legend=dict(orientation="h", y=1.15),
        margin=dict(t=110),
    )
    level_fig.show(config=PLOT_CONFIG)

## 6. Coarse-to-fine generation

Start from $\widehat z^{(0)}=0$. At level $k$, sample all codes in parallel from the predicted categorical distributions, add their contribution, and move to the next scale. After the final level, choose output classes by decoder argmax. Randomness enters through code sampling; no initial noise input is required.

Temperature **1.0** is fixed before evaluation and is not tuned on test results. Entrances/exits are not forced open, and connectivity is not repaired. A standard benchmark sample uses 6 generator passes + 1 decoder pass. The trajectory below additionally decodes after every scale; those extra calls are excluded from benchmark timing.


In [10]:
run.require_trained()
GENERATION_TEMPERATURE = 1.0
EVALUATION_SIZES = (32, 64)
for size in EVALUATION_SIZES:
    _, frames = run.model.sample(
        1,
        size=size,
        temperature=GENERATION_TEMPERATURE,
        seed=SEED + 200 + size,
        capture=True,
    )
    board_panels(
        [frame[0].numpy() for frame in frames],
        [f"Level {k+1}: {s}×{s}" for k, s in enumerate(config.sizes(size))],
        f"NCM · {size}×{size} · generation trajectory",
    ).show(config=PLOT_CONFIG)

## 7. NCM samples and standalone benchmarks

Generate **128 independent NCM mazes** at each resolution and save the raw grids. No other model is loaded or sampled. The same five metric definitions are retained; generation timing includes normal sampling and CPU transfer, excluding preview decoding.

The cache is tied to checkpoint SHA and sampling settings. Results describe this one NCM run, with its actual training budget recorded alongside the outputs.


In [11]:
N_METRIC_SAMPLES = 128
METRIC_BATCH_SIZE = 8
maze_results, maze_evaluation = evaluate_mazes(
    run,
    sizes=EVALUATION_SIZES,
    count=N_METRIC_SAMPLES,
    batch_size=METRIC_BATCH_SIZE,
    temperature=GENERATION_TEMPERATURE,
    artifact_dir=ARTIFACTS,
    seed=SEED,
)
for size, groups in maze_results.items():
    samples = groups["NCM"]
    board_panels(
        samples[:4],
        [f"Sample {i+1}" for i in range(min(4, len(samples)))],
        f"NCM · {size}×{size}",
        columns=4,
    ).show(config=PLOT_CONFIG)
display(
    HTML(
        "<br>".join(
            f"{size}×{size} · NCM: {methods['NCM']:.2f} s / {N_METRIC_SAMPLES} samples"
            for size, methods in maze_evaluation["generation_seconds"].items()
        )
    )
)

IntProgress(value=0, description='Generate mazes', layout=Layout(width='100%'), max=192, style=ProgressStyle(d…

### 7.1 Solvability, A* and passage structure

- **Solvability:** whether a four-neighbor path connects `(0,1)` to `(N−1,N−2)`; a 95% Wilson interval summarizes generated samples.
- **Shortest solution:** shortest path length in moves, among solvable grids only. If no maze is solvable, missing values are not replaced with zeros.
- **A* expansions:** the number of popped and closed cells, including the start and goal; Manhattan heuristic and the same `(f,g,cell)` tie-breaking as FMH. Unsolvable searches run until the frontier is exhausted; a blocked start gives 0. Few expansions alone do not imply a good maze.
- **Reachable passages:** the percentage of all white cells reachable from the entrance; 0% when there are no passage cells.
- **Isolated components:** the number of white components that do not contain the entrance.

Boxplots summarize **generated mazes from one model**. The horizontal mark shows the mean. They do not measure variation across training seeds. Per-sample values are saved in `maze_metrics.csv`.


In [12]:
maze_metric_rows, maze_metric_figures = plot_maze_metrics(maze_results, ARTIFACTS)

IntProgress(value=0, description='Search metrics', layout=Layout(width='100%'), max=256, style=ProgressStyle(d…